In [ ]:
# =====================================================================
# PASO 0: Importación de librerías
# =====================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from fairlearn.metrics import MetricFrame, selection_rate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
# =====================================================================
# PASO 1: Obtención de la información y armado del DataFrame
# =====================================================================

print("Cargando datasets desde INEEd...")
url_cxto = "https://www.ineed.edu.uy/wp-content/uploads/2023/08/Datos_Estudiante_CXTO.csv"
url_socio = "https://www.ineed.edu.uy/wp-content/uploads/2023/08/Datos_Estudiante_Socioemocional.csv"
url_mat = "https://www.ineed.edu.uy/wp-content/uploads/2023/08/Datos_Estudiante_ITEMS_MAT.csv"

# Se agrega el parámetro encoding='latin-1' para poder leer caracteres especiales del español y decimal=',' para que los números con coma sean interpretados correctamente
df_cxto = pd.read_csv(url_cxto, sep=';', encoding='latin-1', decimal=',')
df_socio = pd.read_csv(url_socio, sep=';', encoding='latin-1', decimal=',')
df_mat = pd.read_csv(url_mat, sep=';', encoding='latin-1', decimal=',')

# Unir los dataframes utilizando la variable identificadora
df_merged = df_cxto.merge(df_socio, on="AlumnoCodigoDes", how="inner")
df_merged = df_merged.merge(df_mat, on="AlumnoCodigoDes", how="inner")

# Filtrar a los estudiantes que efectivamente realizaron la prueba
df_final = df_merged[df_merged["IND_Estudiante_ITEMS"] == 1].copy()

# Eliminar registros donde el nivel de matemática sea nulo
df_final = df_final.dropna(subset=["Niveles_MAT"])

In [ ]:
# =====================================================================
# PASO 2: Construcción de dataframes de trabajo (X, y, w)
# =====================================================================

# Diccionario para mapear las variables predictoras
mapa_variables = {
    "EF2d": "ascendencia",
    "AlumnoGenero": "alumno_genero",
    "MdeoInt": "zona",
    "regiones": "region",
    "categoria_centro": "categoria_centro",
    "categoria_grupo": "categoria_grupo",
    "INSE": "idx_inse_alumno",
    "ESCS_Alumno": "idx_esec_alumno",
    "ESCS_Alumno_cat": "idx_esec_alumno_cat",
    "ESCS_Centro": "idx_esec_centro",
    "ESCS_Centro_cat": "idx_esec_centro_cat",
    "ESCS_Grupo": "idx_esec_grupo",
    "ESCS_Grupo_cat": "idx_esec_grupo_cat",
    "ESTSENTPER_E_ESC50": "idx_sentido_pertenencia",
    "VINCENTREEST_E_ESC50": "idx_vinculo_entre_estudiantes",
    "VINESTADS_E_ESC50": "idx_vinculo_estudiantes_adscriptos",
    "VINESTPROF_E_ESC50": "idx_vinculo_estudiantes_docentes",
    "VOZESTU_E_ESC50": "idx_voz_estudiante",
    "ACTITUDMAT_E_ESC50": "idx_actitud_matematica",
    "AUTOCON_E_ESC50": "idx_autocontrol",
    "AUTOEMAT_E_ESC50": "idx_autoeficacia_matematica",
    "AUTOMETA_E_ESC50": "idx_autoregulacion_metacognitiva",
    "EMPATIA_E_ESC50": "idx_empatia",
    "EXTERNALIZ_E_ESC50": "idx_conductas_externalizantes",
    "HABINTER_E_ESC50": "idx_habilidades_interperonales",
    "HABINTRA_E_ESC50": "idx_habilidades_intrapersonales",
    "HABRELAC_E_ESC50": "idx_habilidades_relacionamiento",
    "INTERNALIZ_E_ESC50": "idx_conductas_internalizantes", 
    "MOTAUTREGA_E_ESC50": "idx_motivacion_aprendizaje",
    "MOTINT_E_ESC50": "idx_motivacion_intrinseca",
    "PERSAC_E_ESC50": "idx_perseverancia_academica",
    "REGEMO_E_ESC50": "idx_regulacion_emocional",
    "VALTMAT_E_ESC50": "idx_valoracion_tarea_matematica"
}

# Renombrar columnas para mantener limpieza en el modelo
df_final = df_final.rename(columns=mapa_variables)

# Crear dataset X con las variables mapeadas
variables_modelo = list(mapa_variables.values())
X = df_final[variables_modelo].copy()

# Construir la variable objetivo binaria
# 1 (satisfactorio) = N4 o N5; 0 (insatisfactorio) = N1, N2 o N3
y = np.where(df_final["Niveles_MAT"].isin(["N4", "N5"]), 1, 0)

# Construir el vector de pesos muestrales
w = df_final["peso_MEst"].values

In [ ]:
# =====================================================================
# PASO 3: Preparación previa y split
# =====================================================================

# Split estratificado
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.2, random_state=42, stratify=y
)

# Definición explicita de las columnas categóricas basadas en el diccionario
columnas_categoricas = [
    "ascendencia", "alumno_genero", "zona", "region", 
    "categoria_centro", "categoria_grupo", 
    "idx_esec_alumno_cat", "idx_esec_centro_cat", "idx_esec_grupo_cat"
]

# Las columnas numéricas serán todas las demás de X
columnas_numericas = [col for col in X.columns if col not in columnas_categoricas]

# Por precaución extra, forzamos a numérico (convierte cualquier problema a NaN para que el imputer trabaje)
X_train = X_train.copy()
X_test = X_test.copy()
X_train[columnas_numericas] = X_train[columnas_numericas].apply(pd.to_numeric, errors='coerce')
X_test[columnas_numericas] = X_test[columnas_numericas].apply(pd.to_numeric, errors='coerce')

In [ ]:
# =====================================================================
# PASO 4: Construcción del Preprocesador
# =====================================================================

# Imputación KNN para numéricas y escalado
transformador_num = Pipeline(steps=[
    ("imputer", KNNImputer(n_neighbors=5, weights="distance")),
    ("scaler", StandardScaler()),
])

# Imputación por moda y OneHot para categóricas
transformador_cat = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocesador = ColumnTransformer(transformers=[
    ("num", transformador_num, columnas_numericas),
    ("cat", transformador_cat, columnas_categoricas),
])

In [ ]:
# =====================================================================
# PASO 5: Generación de Modelos, GridSearchCV y Ajuste
# =====================================================================

# Definir los tres modelos
modelos = {
    "Regresion_Logistica": LogisticRegression(random_state=42, class_weight="balanced", max_iter=1000),
    "Random_Forest": RandomForestClassifier(random_state=42, class_weight="balanced"),
    # En XGBoost usamos scale_pos_weight en lugar de class_weight
    "XGBoost": XGBClassifier(random_state=42, eval_metric="auc", scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train))
}

# Definir grillas de hiperparámetros
param_grids = {
    "Regresion_Logistica": {
        "clasificador__C": [0.1, 1.0, 10.0]
    },
    "Random_Forest": {
        "clasificador__n_estimators": [100, 200],
        "clasificador__max_depth": [10, 20, None],
        "clasificador__min_samples_split": [5, 10]
    },
    "XGBoost": {
        "clasificador__n_estimators": [100, 200],
        "clasificador__max_depth": [3, 5, 7],
        "clasificador__learning_rate": [0.01, 0.1]
    }
}

cv_estrategia = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados_grid = {}

# Entrenar todos los modelos
for nombre_modelo, modelo in modelos.items():
    print(f"\nEntrenando {nombre_modelo}...")
    pipeline_completo = Pipeline(steps=[
        ("preprocesamiento", preprocesador),
        ("clasificador", modelo)
    ])
    
    grid_search = GridSearchCV(
        estimator=pipeline_completo,
        param_grid=param_grids[nombre_modelo],
        cv=cv_estrategia,
        scoring="roc_auc",
        n_jobs=-1,
        error_score="raise"
    )
    
    # Ajuste con pesos[cite: 1]
    fit_params = {"clasificador__sample_weight": w_train}
    # Nota: para pipelines con XGBoost o Sklearn moderno, este mapeo de pesos funciona correctamente.
    grid_search.fit(X_train, y_train, **fit_params)
    
    resultados_grid[nombre_modelo] = grid_search

In [ ]:
# =====================================================================
# PASO 6: Definición del modelo ganador
# =====================================================================

mejor_nombre = None
mejor_score = 0
mejor_grid = None

print("\n--- Comparación de Modelos (ROC-AUC en Validación Cruzada) ---")
for nombre, grid in resultados_grid.items():
    score_cv = grid.best_score_
    print(f"{nombre}: {score_cv:.4f}")
    if score_cv > mejor_score:
        mejor_score = score_cv
        mejor_nombre = nombre
        mejor_grid = grid

print(f"\nEl modelo ganador es: {mejor_nombre} con ROC-AUC de {mejor_score:.4f}")
mejor_modelo = mejor_grid.best_estimator_

In [ ]:
# =====================================================================
# PASO 7: Análisis de Feature Importance
# =====================================================================

print("\n--- Feature Importance del Mejor Modelo ---")
try:
    # Extraer el clasificador y los nombres de las features del preprocesador
    clasificador_entrenado = mejor_modelo.named_steps["clasificador"]
    
    # Obtener nombres de columnas categóricas luego del OneHot
    nombres_cat = mejor_modelo.named_steps["preprocesamiento"].named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(columnas_categoricas)
    todas_las_features = columnas_numericas + list(nombres_cat)
    
    if hasattr(clasificador_entrenado, "feature_importances_"):
        importancias = clasificador_entrenado.feature_importances_
    elif hasattr(clasificador_entrenado, "coef_"):
        importancias = np.abs(clasificador_entrenado.coef_[0])
        
    df_importances = pd.DataFrame({"Feature": todas_las_features, "Importancia": importancias})
    df_importances = df_importances.sort_values(by="Importancia", ascending=False).head(10)
    print(df_importances)
except Exception as e:
    print("No se pudo calcular la importancia de variables directamente. (Revisar compatibilidad del modelo).")

In [ ]:
# =====================================================================
# PASO 8: Resultados y evaluación en Test
# =====================================================================

print("\n--- Evaluación en Test (Ponderada) ---")
print("Mejores hiperparámetros encontrados:", mejor_grid.best_params_)

y_pred_proba = mejor_modelo.predict_proba(X_test)[:, 1]
score_test = roc_auc_score(y_test, y_pred_proba, sample_weight=w_test)
print(f"ROC-AUC en conjunto de Test (ponderado): {score_test:.4f}")

# Generar la predicción estándar de clases
y_pred = mejor_modelo.predict(X_test)

# Calcular la matriz de confusión PONDERADA
cm_ponderada = confusion_matrix(y_test, y_pred, sample_weight=w_test)

# Configurar Numpy para evitar notación científica en el print de la consola
np.set_printoptions(suppress=True)
print("Matriz de Confusión Ponderada (Poblacional):")
print(cm_ponderada)

# Visualizarla de forma gráfica evitando notación científica
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_ponderada, 
    display_labels=["Insatisfactorio", "Satisfactorio"]
)

# El parámetro values_format=',.0f' quita la notación científica y agrega separador de miles
disp.plot(cmap=plt.cm.Blues, values_format=',.0f')
plt.title(f"Matriz de Confusión Ponderada - {mejor_nombre}")
plt.show()

In [ ]:
# =====================================================================
# PASO 9: Auditoría de Equidad (Fairness)
# =====================================================================

print("\n" + "="*60)
print("--- Auditoría de Equidad (Fairness) ---")
print("="*60)

variables_auditar = ["ascendencia", "alumno_genero", "zona", "categoria_centro"]

# Diccionario actualizado con el orden específico solicitado
metricas_a_evaluar = {
    "Accuracy": accuracy_score,
    "Precision": precision_score,
    "Recall": recall_score,
    "F1-score": f1_score,
    "ROC AUC": roc_auc_score
}

for var in variables_auditar:
    # Se extrae la variable sensible del conjunto de test original
    variable_sensible_test = X_test[var]
    
    metric_frame = MetricFrame(
        metrics=metricas_a_evaluar,
        y_true=y_test,
        y_pred=y_pred,
        sensitive_features=variable_sensible_test,
        sample_params={
            "Accuracy": {"sample_weight": w_test},
            "Precision": {"sample_weight": w_test},
            "Recall": {"sample_weight": w_test},
            "F1-score": {"sample_weight": w_test},
            "ROC AUC": {"sample_weight": w_test},
        },
    )
    
    # Obtener el dataframe de resultados
    df_resultados = metric_frame.by_group
    
    # 1. SALIDA EN CONSOLA MEJORADA
    print(f"\n>> Resultados poblacionales por subgrupo de: {var.upper()}")
    print("-" * 60)
    # Formatear la tabla para mostrar exactamente 3 decimales
    print(df_resultados.to_string(float_format="{:.3f}".format))
    print("-" * 60)
    
    # 2. VISUALIZACIÓN GRÁFICA
    # Genera un gráfico de barras agrupadas para comparar fácilmente
    ax = df_resultados.plot(
        kind='bar', 
        figsize=(10, 5), 
        colormap='viridis', # Una paleta de colores elegante y accesible
        edgecolor='black',
        alpha=0.85
    )
    
    plt.title(f"Métricas de Equidad por subgrupo: {var.capitalize()}", fontsize=14, pad=15)
    plt.ylabel("Valor de la Métrica", fontsize=12)
    plt.xlabel(var.capitalize(), fontsize=12)
    plt.xticks(rotation=0) # Mantener los nombres de las etiquetas horizontales
    
    # Mover la leyenda fuera del gráfico para que no tape las barras
    plt.legend(title="Métricas", bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Añadir el valor numérico encima de cada barra
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', padding=3, fontsize=9)
        
    plt.tight_layout()
    plt.show()